# Fire up libraries

In [117]:
import matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn import linear_model
from random import randint

# Criando bases de dados


In [11]:
dados = pd.read_csv('train1.csv')

In [12]:
dados.head()

,ip,app,device,os,channel,click_time,attributed_time,is_attributed
0,83230,3,1,13,379,2017-11-06 14:32:21,NaN,0
1,17357,3,1,19,379,2017-11-06 14:33:34,NaN,0
2,35810,3,1,13,379,2017-11-06 14:34:12,NaN,0
3,45745,14,1,13,478,2017-11-06 14:34:52,NaN,0
4,161007,3,1,13,379,2017-11-06 14:35:08,NaN,0


In [13]:
dados.shape

(2000000, 8)

In [14]:
dados[dados['is_attributed']==1].shape

(3388, 8)

In [56]:
fracao = dados[dados['is_attributed']==1].shape[0]*1.0/dados.shape[0]
print fracao*100,'% das entradas pertencem a classe 1';

0.1694 % das entradas pertencem a classe 1


## Base de dados 1 (Dados balanceados)

In [57]:
fracao = dados[dados['is_attributed']==1].shape[0]*1.0/dados[dados['is_attributed']==0].shape[0]
classe0 = dados[dados['is_attributed']==0]
classe1 = dados[dados['is_attributed']==1]
classe0b = classe0.sample(frac=fracao, random_state=200)
classe0b.shape

(3388, 8)

In [61]:
classe0b.head()

,ip,app,device,os,channel,click_time,attributed_time,is_attributed
172127,25321,21,1,19,128,2017-11-06 16:03:13,NaN,0
1955972,117867,2,1,53,205,2017-11-06 16:48:27,NaN,0
1991732,73487,8,1,13,145,2017-11-06 16:49:35,NaN,0
1215731,30564,12,1,19,178,2017-11-06 16:27:33,NaN,0
1715,157379,3,1,6,19,2017-11-06 16:00:02,NaN,0


In [62]:
classe1.head()

,ip,app,device,os,channel,click_time,attributed_time,is_attributed
103,204158,35,1,13,21,2017-11-06 15:41:07,2017-11-07 08:17:19,1
1504,29692,9,1,22,215,2017-11-06 16:00:02,2017-11-07 10:05:22,1
1798,64516,35,1,13,21,2017-11-06 16:00:02,2017-11-06 23:40:50,1
2102,172429,35,1,46,274,2017-11-06 16:00:03,2017-11-07 00:55:29,1
3056,199085,35,1,13,274,2017-11-06 16:00:04,2017-11-06 23:04:54,1


In [83]:
pieces = [classe0b, classe1]
bd1 = pd.concat(pieces)
bd1 = bd1.sample(frac=1, random_state=200).reset_index(drop=True)
bd1.head()

,ip,app,device,os,channel,click_time,attributed_time,is_attributed
0,49600,107,1,17,171,2017-11-06 16:28:17,2017-11-07 02:39:44,1
1,43210,1,1,19,452,2017-11-06 16:05:28,NaN,0
2,125232,15,1,15,245,2017-11-06 16:48:42,NaN,0
3,75979,35,1,41,274,2017-11-06 16:07:07,2017-11-06 18:57:58,1
4,1415,9,1,23,134,2017-11-06 16:23:11,2017-11-07 04:17:49,1


In [84]:
bd1.shape

(6776, 8)

In [88]:
fracao = bd1[bd1['is_attributed']==1].shape[0]*1.0/bd1.shape[0]
print fracao*100,'% das entradas pertencem a classe 1';

50.0 % das entradas pertencem a classe 1


## Base de dados 2

In [114]:
fracao = bd1.shape[0]*1.0/dados.shape[0]
bd2 = dados.sample(frac=fracao, random_state=200)
bd2 = bd2.sample(frac=1, random_state=200).reset_index(drop=True)
bd2.shape

(6776, 8)

In [89]:
fracao = bd2[bd2['is_attributed']==1].shape[0]*1.0/bd2.shape[0]
print fracao*100,'% das entradas pertencem a classe 1';

0.162337662338 % das entradas pertencem a classe 1


# Baseline 1 (majoritário)

## Base de dados 1

In [94]:
cont = 0
recall = 0
acuracia = 0
prev = 1

if(bd1[bd1['is_attributed']==1].shape[0]*1.0/bd1.shape[0] < 0.5):
    prev = 0

for i in range(bd1.shape[0]):
    if(bd1.is_attributed[i] == prev):
        cont = cont + 1

acuracia = cont*1.0/bd1.shape[0]
precision = acuracia
recall = 1

if(prev == 0):
    precision = 0
    recall = 0

print 'acurácia:', acuracia
print 'precisão:', precision
print 'revocação:', recall

acurácia: 0.5
precisão: 0.5
revocação: 1


## Base de dados 2

In [116]:
cont = 0
recall = 0
acuracia = 0
prev = 1

if(bd2[bd2['is_attributed']==1].shape[0]*1.0/bd2.shape[0] < 0.5):
    prev = 0

for i in range(bd2.shape[0]):
    if(bd2.is_attributed[i] == prev):
        cont = cont + 1

acuracia = cont*1.0/bd2.shape[0]
precision = acuracia
recall = 1

if(prev == 0):
    precision = 0
    recall = 0

print 'acurácia:', acuracia
print 'precisão:', precision
print 'revocação:', recall

acurácia: 0.998376623377
precisão: 0
revocação: 0


# Baseline 2 (aleatório)

# Base de dados 1

In [124]:
acuracia = 0
recall = 0
precision = 0

for i in range(bd1.shape[0]):
    aleatorio = randint(0,1)
    if(aleatorio == 1):
        precision = precision + 1
        if(bd1.is_attributed[i] == aleatorio):
            acuracia = acuracia + 1
            recall = recall + 1
    elif(bd1.is_attributed[i] == aleatorio):
        acuracia = acuracia + 1

acuracia = acuracia*1.0/bd1.shape[0]
precision = recall*1.0/precision
recall = recall*1.0/bd1[bd1['is_attributed'] == 1].shape[0]

print 'acurácia:', acuracia
print 'precisão:', precision
print 'revocação:', recall

acurácia: 0.499262101535
precisão: 0.499260573795
revocação: 0.498229043684


## Base de dados 2

In [125]:
acuracia = 0
recall = 0
precision = 0

for i in range(bd2.shape[0]):
    aleatorio = randint(0,1)
    if(aleatorio == 1):
        precision = precision + 1
        if(bd2.is_attributed[i] == aleatorio):
            acuracia = acuracia + 1
            recall = recall + 1
    elif(bd2.is_attributed[i] == aleatorio):
        acuracia = acuracia + 1

acuracia = acuracia*1.0/bd2.shape[0]
precision = recall*1.0/precision
recall = recall*1.0/bd2[bd2['is_attributed'] == 1].shape[0]

print 'acurácia:', acuracia
print 'precisão:', precision
print 'revocação:', recall

acurácia: 0.498229043684
precisão: 0.000883652430044
revocação: 0.272727272727


## Build the regression model using only sqft_living as a feature

In [ ]:
x_train = train_data['sqft_living'].values.reshape(-1,1)
y_train = train_data['price'].values.reshape(-1,1)

In [ ]:
simple_model = linear_model.LinearRegression()
simple_model.fit(x_train, y_train)

In [ ]:
plt.grid('on')
plt.scatter(x_train, y_train)
plt.plot(x_train, simple_model.predict(x_train), color='red', linewidth=2)
plt.show()

# Let's show what our predictions look like

In [ ]:
x_test = test_data['sqft_living'].values.reshape(-1,1)
y_test = test_data['price'].values.reshape(-1,1)

y_pred = simple_model.predict(x_test)

In [ ]:
plt.grid('on')
plt.scatter(x_test, y_test)
plt.plot(x_test,y_pred, color='red', linewidth=2)
plt.show()

# Evaluate the simple model

In [ ]:
def rmse(predictions, targets):
    return np.sqrt(((predictions - targets) ** 2).mean())

In [ ]:
print('intercept:', simple_model.intercept_, 'coefficients:', simple_model.coef_)
# The mean squared error
print("RMSE: %.2f" % (rmse(y_pred, y_test)))

RMSE of about $254.323,39

# Explore other features in the data

To build a more elaborate model, we will explore using more features.

In [ ]:
my_features = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'zipcode']

In [ ]:
sales[my_features].describe()

In [ ]:
#sales.show(view='BoxWhisker Plot', x='zipcode', y='price')
plt.grid('on')
plt.scatter(sales['zipcode'], sales['price'])
plt.show()

98039 is the most expensive zip code.

# Build a regression model with more features

In [ ]:
#my_features_model = (train_data,target='price',features=my_features,validation_set=None)
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['price'].values.reshape(-1,1)

In [ ]:
mult_model = linear_model.LinearRegression()
mult_model.fit(x_train, y_train)

## Comparing the results of the simple model with adding more features

In [ ]:
x_test = test_data[my_features].values.reshape(-1,len(my_features))
y_test = test_data['price'].values.reshape(-1,1)

y_pred = mult_model.predict(x_test)

In [ ]:
print('intercept:', mult_model.intercept_, 'coefficients:', mult_model.coef_)
# The mean squared error
print("RMSE: %.2f" % (rmse(y_pred, y_test)))

The RMSE goes down from \$254.323,39 to \$228.024,43 with more features.

# Apply learned models to predict prices of 3 houses

The first house we will use is considered an "average" house in Seattle. 

In [ ]:
house1 = sales[sales['id']==5309101200]

In [ ]:
house1

<img src="house1.png">

In [ ]:
print house1['price']

In [ ]:
print simple_model.predict(house1['sqft_living'])

In [ ]:
print mult_model.predict(house1[my_features])

In this case, the model with more features provides a worse prediction than the simpler model with only 1 feature.  However, on average, the model with more features is better.

## Prediction for a second, fancier house

We will now examine the predictions for a fancier house.

In [ ]:
house2 = sales[sales['id']==1925069082]

In [ ]:
house2

<img src="house2.jpg">

In [ ]:
print house2['price']

In [ ]:
print simple_model.predict(house2['sqft_living'].values.reshape(-1,1))

In [ ]:
print mult_model.predict(house2[my_features])

In this case, the model with more features provides a better prediction.  This behavior is expected here, because this house is more differentiated by features that go beyond its square feet of living space, especially the fact that it's a waterfront house. 

## Last house, super fancy

Our last house is a very large one owned by a famous Seattleite.

In [ ]:
bill_gates = {'bedrooms':[8], 
              'bathrooms':[25], 
              'sqft_living':[50000], 
              'sqft_lot':[225000],
              'floors':[4], 
              'zipcode':['98039'], 
              'condition':[10], 
              'grade':[10],
              'waterfront':[1],
              'view':[4],
              'sqft_above':[37500],
              'sqft_basement':[12500],
              'yr_built':[1994],
              'yr_renovated':[2010],
              'lat':[47.627606],
              'long':[-122.242054],
              'sqft_living15':[5000],
              'sqft_lot15':[40000]}

<img src="house_bill.jpg">

In [ ]:
print simple_model.predict(pd.DataFrame(bill_gates)['sqft_living'].values.reshape(-1,1))

The model predicts a price of over $14M for this house! But we expect the house to cost much more.  (There are very few samples in the dataset of houses that are this fancy, so we don't expect the model to capture a perfect prediction here.)

In [ ]:
print mult_model.predict(pd.DataFrame(bill_gates)[my_features])

# Answers


# 1 - Selection and summary statistics

# 2 - Filtering data

# 3 - Building a regression model with several more features